# Strategy Correlation Matrix

Pairwise return correlation across every strategy's genuine backtest equity curve under `outputs/<strategy_id>/`, plus the S&P 500 (SPY) benchmark. This notebook only ever reads real, already-computed equity curves -- it never fabricates, backfills, or simulates a return series for a strategy that hasn't actually been backtested.

**Extending this as new strategies are added:** write a strategy's backtest equity curve to `outputs/<strategy_id>/<strategy_id>_equity_curve.csv` with a `quarter_end` column and a `<strategy_id>_equity` column (the same shape `outputs/filing_momentum_ml/filing_momentum_ml_equity_curve.csv` already has). This notebook auto-discovers every `outputs/*/*_equity_curve.csv` file on each run -- no code change needed here to add a strategy to the matrix. Currently only Filing Momentum ML has a real backtest equity curve; Ranked Multi-Factor Rotation is scaffold-only (no formulas or backtest yet), so it does not appear below.

In [1]:
from pathlib import Path
import sys


def resolve_project_root() -> Path:
    candidates = []
    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass
    candidates.append(Path("/Users/gavinhussey/Downloads/AtlasQuant"))
    for candidate in candidates:
        if candidate.name == "outputs":
            candidate = candidate.parent
        if (candidate / "src" / "atlas_quant").exists():
            return candidate
    raise FileNotFoundError("Could not locate AtlasQuant project root")


PROJECT_ROOT = resolve_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MATRIX_CSV = OUTPUT_DIR / "strategy_correlation_matrix.csv"

# Every file here is treated as one strategy's genuine backtest equity curve,
# never a placeholder. Add a strategy to this matrix by writing
# outputs/<strategy_id>/<strategy_id>_equity_curve.csv with a "quarter_end"
# column and a "<strategy_id>_equity" column -- no code change needed below.
EQUITY_CURVE_CSVS = sorted(OUTPUT_DIR.glob("*/*_equity_curve.csv"))
BENCHMARK_COLUMN = "sp500_equity"
BENCHMARK_LABEL = "S&P 500 (SPY)"

print("project root:", PROJECT_ROOT)
print("equity curve files found:")
for p in EQUITY_CURVE_CSVS:
    print(" -", p.name)


project root: /Users/gavinhussey/Downloads/AtlasQuant
equity curve files found:
 - filing_momentum_ml_equity_curve.csv


In [2]:
import pandas as pd

if not EQUITY_CURVE_CSVS:
    raise FileNotFoundError(
        f"no *_equity_curve.csv files found under {OUTPUT_DIR} -- "
        "run a strategy's backtest and export its equity curve first."
    )

series_frames = []
benchmark_frame = None

for csv_path in EQUITY_CURVE_CSVS:
    df = pd.read_csv(csv_path)
    if "quarter_end" not in df.columns:
        raise ValueError(f"{csv_path.name} is missing a 'quarter_end' column")
    df["quarter_end"] = pd.to_datetime(df["quarter_end"])

    equity_columns = [c for c in df.columns if c.endswith("_equity") and c != BENCHMARK_COLUMN]
    if not equity_columns:
        raise ValueError(f"{csv_path.name} has no '<strategy_id>_equity' column to include")
    for col in equity_columns:
        series_frames.append(df[["quarter_end", col]].set_index("quarter_end"))

    if BENCHMARK_COLUMN in df.columns and benchmark_frame is None:
        # Every strategy backtest resolves SPY the same way (see
        # atlas_quant.backtest.benchmark) -- the first file's benchmark
        # column is used; later files are expected to agree, not re-merged.
        benchmark_frame = df[["quarter_end", BENCHMARK_COLUMN]].set_index("quarter_end")

frames = series_frames + ([benchmark_frame] if benchmark_frame is not None else [])
equity = pd.concat(frames, axis=1).sort_index()
equity.tail()


,filing_momentum_ml_equity,sp500_equity
quarter_end,,
2025-06-30,11.899246,3.192649
2025-09-30,13.115784,3.244090
2025-12-31,17.521835,3.470233
2026-03-31,17.521835,3.470233
2026-06-30,17.521835,3.470233


In [3]:
# Quarter-over-quarter simple returns from each equity curve. Rows before a
# strategy took its first position, and any trailing quarters beyond a
# backtest's current end date, are flat (0.0) by construction -- real
# reported values, not synthetic, but see the caveat below before reading
# too much into the correlation they produce.
returns = equity.pct_change().dropna(how="all")

label_map = {
    c: (BENCHMARK_LABEL if c == BENCHMARK_COLUMN else c.removesuffix("_equity"))
    for c in returns.columns
}
returns = returns.rename(columns=label_map)

correlation = returns.corr()
correlation.to_csv(MATRIX_CSV)
print(f"wrote {MATRIX_CSV} ({len(returns)} quarterly observations)")
correlation


wrote /Users/gavinhussey/Downloads/AtlasQuant/outputs/strategy_correlation_matrix.csv (45 quarterly observations)


,filing_momentum_ml,S&P 500 (SPY)
filing_momentum_ml,1.000000,0.750835
S&P 500 (SPY),0.750835,1.000000


**Caveat:** quarters where a strategy held no positions yet (Filing Momentum ML's 2015-2016 ramp-up, before enough training history existed) or falls beyond its backtest's current end date report a flat 0.0 return in the source CSV. Those are genuine reported values, not fabricated, but simultaneous (0.0, 0.0) pairs across series mechanically push correlation *up* rather than reflecting real co-movement. `n_quarters` above is the full observation count including these; narrow `EQUITY_CURVE_CSVS`/trim `equity` to each strategy's active window first if a stricter, ramp-up-excluded correlation is needed.

In [4]:
from html import escape
from IPython.display import HTML, display


def _color_for(value: float) -> str:
    # Diverging scale: -1 -> blue, 0 -> white, +1 -> red.
    v = max(-1.0, min(1.0, value))
    if v >= 0:
        r = 255 + v * (220 - 255)
        g = 255 + v * (38 - 255)
        b = 255 + v * (38 - 255)
    else:
        t = -v
        r = 255 + t * (37 - 255)
        g = 255 + t * (99 - 255)
        b = 255 + t * (235 - 255)
    return f"rgb({r:.0f},{g:.0f},{b:.0f})"


def render_correlation_heatmap(corr: pd.DataFrame) -> str:
    labels = list(corr.columns)
    header = "".join(f'<th style="padding:8px 14px;text-align:right;font-size:13px;color:#374151;">{escape(l)}</th>' for l in labels)
    rows_html = []
    for row_label in labels:
        cells_html = []
        for col_label in labels:
            v = corr.loc[row_label, col_label]
            bg = _color_for(v)
            text_color = "#111827" if abs(v) < 0.6 else "#ffffff"
            cells_html.append(
                f'<td style="padding:8px 14px;text-align:center;background:{bg};color:{text_color};font-variant-numeric:tabular-nums;">{v:+.2f}</td>'
            )
        rows_html.append(
            f'<tr><th style="padding:8px 14px;text-align:left;font-size:13px;color:#374151;">{escape(row_label)}</th>{"".join(cells_html)}</tr>'
        )
    return f'''
    <table style="border-collapse:collapse;font-family:-apple-system,Helvetica,Arial,sans-serif;border:1px solid #d1d5db;">
      <thead><tr><th></th>{header}</tr></thead>
      <tbody>{"".join(rows_html)}</tbody>
    </table>
    '''


display(HTML(render_correlation_heatmap(correlation)))


,filing_momentum_ml,S&P 500 (SPY)
filing_momentum_ml,+1.00,+0.75
S&P 500 (SPY),+0.75,+1.00
